# Classify Intents with an LLM, and Compare

Yesterday, classic models, *embeddings* and a fine-tuned encoder classified MASSIVE's requests. Today, an LLM, without training:

1. Zero-shot: the list of described intents, a constrained output
2. With a few examples in the prompt
3. Compare every method
4. Read the errors

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, a smaller LLM (Qwen3-1.7B instead of Qwen3-4B) runs on the CPU, more slowly: it then classifies 10 requests instead of 300 (about seven minutes for both prompts). Enough to write and run the code, not to conclude: on 10 requests, and with another model, your F1 in exercise 2 doesn't tell whether the examples help. Exercise 3 then loads Qwen3-4B's results on the 300 requests, computed on a GPU by the same code: the table, and the answer to exercise 2, are the GPU's. Exercise 4 covers the whole test set, with or without a GPU.

## Data and Model

Yesterday's results: upload your files `predictions-en.csv` and `timings-en.csv` (*Files* panel on the left), or use ours, which the next cell downloads if they're missing. Ollama installs and downloads the model in the background.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh massive-intents
!test -f predictions-en.csv || cp massive-intents/lab-results/predictions-en.csv .
!test -f timings-en.csv || cp massive-intents/lab-results/timings-en.csv .
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# On a GPU, Qwen3-4B. On a CPU, Qwen3-1.7B, twice as fast: a "hybrid" model, which
# thinks before answering, except with reasoning_effort="none" (in the calls).
MODEL = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# On a GPU, 8 requests at a time; on a CPU, one: the prompts' common start (the
# instructions) is then computed only once.
parallel = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": parallel, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
pull = subprocess.Popen(
  f"sleep 5 && ollama pull {MODEL}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Literal

import pandas as pd
from openai import OpenAI
from pydantic import create_model
from sklearn.metrics import f1_score

train = pd.read_parquet("massive-intents/en-train.parquet")
intents = pd.read_csv("massive-intents/intents.csv")
test = pd.read_parquet("massive-intents/en-test.parquet")
# The requests' texts, by their id.
predictions = pd.read_csv("predictions-en.csv", dtype={"id": str}).merge(
  test[["id", "text"]], on="id"
)
timings = pd.read_csv("timings-en.csv")
# 300 test requests, drawn at random: an LLM is too slow for all 2,974. Without a GPU,
# 10: each request takes several seconds there.
sample = predictions.sample(300 if GPU else 10, random_state=0).reset_index(drop=True)
# Each method's macro F1 covers the intents of `sample`: otherwise, a prediction of
# an absent intent would add a class with an F1 of 0 to it.
sample_labels = sorted(sample.label.unique())
labels = intents.label.tolist()
print(len(labels), "intents")

assert pull.wait() == 0, "Model download failed"
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# A first call loads the model (onto the GPU if there is one), where it stays: before any parallel work.
client.chat.completions.create(
  model=MODEL, messages=[{"role": "user", "content": "Hello"}], max_tokens=1
)
print(MODEL, "ready")

## 1. Zero-Shot

The output schema, `Intent`, has a single field, `intent`, whose value is necessarily one of the 60 labels: the model can't answer outside the list.

Write the system prompt (`ZERO_SHOT`): the task, then the list of intents with their description (`intents.description_en`). Write `classify(text, system)` that returns the chosen intent, classify `sample`'s requests (300; without a GPU, 10) in parallel (with a function `run(system)`, which the rest of the lab reuses: store its results in `zero_shot` and `zero_seconds`), and measure the macro F1 and the time per text.

**Useful documentation**

- [`pydantic.create_model`](https://docs.pydantic.dev/latest/concepts/models/#dynamic-model-creation)
- [`DataFrame.itertuples`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)

In [ ]:
Intent = create_model("Intent", intent=(Literal[tuple(labels)], ...))

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `intents.itertuples()`: `row.label` and `row.description_en`; one `- label: description` line per intent, joined with `"\n"`
- the prompt: the task ("you classify requests made to a voice assistant by their intent"), then "choose exactly one intent from this list" and the list
- `client.chat.completions.parse(model=MODEL, messages=[{"role": "system", "content": system}, {"role": "user", "content": text}], response_format=Intent, temperature=0, reasoning_effort="none")`; the intent is in `response.choices[0].message.parsed.intent`
- `reasoning_effort="none"`: the model answers without a thinking phase (without a GPU, Qwen3-1.7B would have one, which multiplies the response time)
- `ThreadPoolExecutor(8)`: 8 calls in parallel; `pool.map(function, sample.text)` returns the results in the texts' order
- `f1_score(sample.label, predictions, average="macro", labels=sample_labels)`: `average="macro"` gives every class the same weight, `labels` limits the average to the intents of `sample`; the time per text in ms: `1000 * seconds / len(sample)`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
LIST = "\n".join(f"- {row.label}: {row.description_en}" for row in ...)
ZERO_SHOT = f"""You classify requests made to a voice assistant by their intent.
Choose exactly one intent from this list:
{LIST}"""


def classify(text: str, system: str) -> str:
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
  )
  return response.choices[0].message.parsed.intent


def run(system: str) -> tuple[list[str], float]:
  start = time.time()
  with ThreadPoolExecutor(8) as pool:
    found = list(pool.map(lambda text: classify(text, system), ...))  # `sample`'s texts
  return found, time.time() - start


zero_shot, zero_seconds = run(ZERO_SHOT)
print(
  f"macro F1 {f1_score(sample.label, ..., average=..., labels=sample_labels):.3f}, {1000 * zero_seconds / len(sample):.0f} ms/text"
)
```

### Solution

In [ ]:
LIST = "\n".join(f"- {row.label}: {row.description_en}" for row in intents.itertuples())
ZERO_SHOT = f"""You classify requests made to a voice assistant by their intent.
Choose exactly one intent from this list:
{LIST}"""


def classify(text: str, system: str) -> str:
  response = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
    response_format=Intent,
    temperature=0,
    reasoning_effort="none",
  )
  return response.choices[0].message.parsed.intent


def run(system: str) -> tuple[list[str], float]:
  start = time.time()
  with ThreadPoolExecutor(8) as pool:
    found = list(pool.map(lambda text: classify(text, system), sample.text))
  return found, time.time() - start


zero_shot, zero_seconds = run(ZERO_SHOT)
print(
  f"macro F1 {f1_score(sample.label, zero_shot, average='macro', labels=sample_labels):.3f}, {1000 * zero_seconds / len(sample):.0f} ms/text"
)

# On a GPU (Qwen3-4B, 300 requests, an RTX A4000): macro F1 0.754, 302 ms per text.
# Without a GPU, on 10 requests and with Qwen3-1.7B, this F1 doesn't compare.

## 2. With a Few Examples

Add two training examples per intent to the prompt. Does the F1 improve? How many tokens does the prompt have now, against how many without examples, and what would it cost through an API, at $0.10 per million input tokens, for 1 million requests?

Without a GPU, your F1 doesn't answer the first question: on 10 requests, one error more or less moves it by about ten points, and Qwen3-1.7B isn't the model that runs on a GPU. The answer is in exercise 3's table, whose first cell loads the GPU's results on the 300 requests. The tokens and the cost depend only on the prompt.

**Useful documentation**

- [`DataFrameGroupBy.sample`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.sample.html)
- [Ollama: OpenAI compatibility](https://docs.ollama.com/api/openai-compatibility)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `train.groupby("label").sample(2, random_state=0)`: 2 examples per intent, always the same ones
- one example per line, after the previous exercise's prompt: `"text" -> intent`
- `run(FEW_SHOT)`: the previous exercise's function, with the new prompt
- `response.usage.prompt_tokens`: the prompt's number of tokens, in the response of a `client.chat.completions.create(..., max_tokens=1)` call (a single generated token is enough), for each of the two prompts

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
examples = train.groupby("label").sample(2, random_state=0)
EXAMPLES = "\n".join(f'"{row.text}" -> {row.label}' for row in ...)
FEW_SHOT = f"{ZERO_SHOT}\n\nExamples:\n{EXAMPLES}"

few_shot, few_seconds = run(...)
print(
  f"macro F1 {f1_score(sample.label, few_shot, average='macro', labels=sample_labels):.3f}, {1000 * few_seconds / len(sample):.0f} ms/text"
)

for method, system in (("LLM zero-shot", ZERO_SHOT), ("LLM few-shot", FEW_SHOT)):
  usage = client.chat.completions.create(
    model=MODEL,
    messages=[
      {"role": "system", "content": ...},
      {"role": "user", "content": sample.text[0]},
    ],
    max_tokens=1,
  ).usage
  # 1 million requests of prompt_tokens tokens, at $0.10 per million tokens:
  print(
    f"{method}: {usage.prompt_tokens} prompt tokens,",
    f"${... * 0.10:.0f} for 1 million requests",
  )
```

### Solution

In [ ]:
examples = train.groupby("label").sample(2, random_state=0)
EXAMPLES = "\n".join(f'"{row.text}" -> {row.label}' for row in examples.itertuples())
FEW_SHOT = f"{ZERO_SHOT}\n\nExamples:\n{EXAMPLES}"

few_shot, few_seconds = run(FEW_SHOT)
print(
  f"macro F1 {f1_score(sample.label, few_shot, average='macro', labels=sample_labels):.3f}, {1000 * few_seconds / len(sample):.0f} ms/text"
)

for method, system in (("LLM zero-shot", ZERO_SHOT), ("LLM few-shot", FEW_SHOT)):
  usage = client.chat.completions.create(
    model=MODEL,
    messages=[
      {"role": "system", "content": system},
      {"role": "user", "content": sample.text[0]},
    ],
    max_tokens=1,
  ).usage
  # 1 million requests of prompt_tokens tokens, at $0.10 per million tokens:
  print(
    f"{method}: {usage.prompt_tokens} prompt tokens,",
    f"${usage.prompt_tokens * 0.10:.0f} for 1 million requests",
  )

# A long shared prefix (instructions + examples) is exactly what the APIs' prompt caching
# bills at a discount.
# On a GPU (Qwen3-4B, 300 requests): macro F1 0.763 few-shot against 0.754 zero-shot,
# 331 ms per text against 302, and 2,426 prompt tokens against 751 ($243 against $75
# for 1 million requests). One more point of F1, on 300 requests, doesn't establish
# that the examples help, for three times the tokens.

## 3. Compare Every Method

Gather in one table, on the **same** requests of `sample`: the macro F1 of each of yesterday's methods (`sample`'s columns) and of today's two, their training time and their time per text (`timings`). Which one would you put in production for 10 million requests a month? And for a prototype due tomorrow?

Three caveats before reading the table:

- **The times don't come from the same hardware.** The TF-IDF methods ran on a CPU, the *embeddings* and the encoder on yesterday's GPU (yours, or the one of our published results), the LLM on this notebook's GPU (without one, the one of our published results). On the same hardware, a 2-core CPU, the fine-tuned encoder takes more than a hundred ms per text.
- **The F1 isn't yesterday's.** It covers 300 requests instead of 2,974, and only the intents they contain: with our results, 0.883 for the encoder, against 0.869 yesterday. From one draw of 300 requests to another, it moves by about ±5 points: two methods a few points apart can't be told apart.
- **Without a GPU**, the next cell replaces your 10 requests' results by Qwen3-4B's on the 300, computed on a GPU by the same code: `sample`, `zero_shot`, `few_shot` and their times. The table then covers the 300 requests, as on a GPU.

**Useful documentation**

- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [`DataFrame.set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html)

In [ ]:
# Without a GPU, exercises 1 and 2 only classified 10 requests, with Qwen3-1.7B: we load
# Qwen3-4B's predictions on the 300, computed on a GPU by the same code, and their
# times.
if not GPU:
  sample = predictions.sample(300, random_state=0).reset_index(drop=True)
  sample_labels = sorted(sample.label.unique())
  llm = pd.read_csv(
    "massive-intents/lab-results/llm-predictions-en.csv", dtype={"id": str}
  )
  assert llm.id.tolist() == sample.id.tolist(), "Not `sample`'s requests"
  zero_shot, few_shot = llm.zero_shot.tolist(), llm.few_shot.tolist()
  llm_timings = pd.read_csv("massive-intents/lab-results/llm-timings-en.csv")
  ms_per_text = llm_timings.set_index("method").ms_per_text
  zero_seconds = ms_per_text["LLM zero-shot"] * len(sample) / 1000
  few_seconds = ms_per_text["LLM few-shot"] * len(sample) / 1000
  print("GPU results loaded:", len(sample), "requests, qwen3:4b-instruct-2507")

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- yesterday's methods are `sample`'s columns other than `id`, `label` and `text`
- `timings.set_index("method").loc[method]`: a method's timing row (`.train_s`, `.ms_per_text`)
- `f1_score(sample.label, sample[method], average="macro", labels=sample_labels)`: `average="macro"` gives every class the same weight, `labels` limits the average to the intents of `sample`
- today's two methods: `zero_shot` and `few_shot`, with no training (0 s), and their time per text `1000 * seconds / len(sample)`
- `pd.DataFrame(rows, columns=[...]).round(3)` from a list of tuples

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
methods = [column for column in sample.columns if column not in ("id", "label", "text")]
rows = []
for method in methods:
  timing = timings.set_index("method").loc[...]
  rows.append(
    (
      method,
      f1_score(sample.label, sample[...], average="macro", labels=sample_labels),
      timing.train_s,
      timing.ms_per_text,
    )
  )
rows.append(
  (
    "LLM zero-shot",
    f1_score(sample.label, ..., average="macro", labels=sample_labels),
    0,
    1000 * zero_seconds / len(sample),
  )
)
rows.append(
  (
    "LLM few-shot",
    f1_score(sample.label, ..., average="macro", labels=sample_labels),
    0,
    1000 * few_seconds / len(sample),
  )
)
pd.DataFrame(rows, columns=["method", "macro F1", "training (s)", "ms/text"]).round(3)
```

### Solution

In [ ]:
methods = [column for column in sample.columns if column not in ("id", "label", "text")]
rows = []
for method in methods:
  timing = timings.set_index("method").loc[method]
  rows.append(
    (
      method,
      f1_score(sample.label, sample[method], average="macro", labels=sample_labels),
      timing.train_s,
      timing.ms_per_text,
    )
  )
rows.append(
  (
    "LLM zero-shot",
    f1_score(sample.label, zero_shot, average="macro", labels=sample_labels),
    0,
    1000 * zero_seconds / len(sample),
  )
)
rows.append(
  (
    "LLM few-shot",
    f1_score(sample.label, few_shot, average="macro", labels=sample_labels),
    0,
    1000 * few_seconds / len(sample),
  )
)
pd.DataFrame(rows, columns=["method", "macro F1", "training (s)", "ms/text"]).round(3)

# Times only compare on the same hardware: on a GPU, the fine-tuned encoder takes under
# a ms per text; on a CPU, more than a hundred, and the LLM several seconds. In
# production at high volume: the fine-tuned encoder, or the embeddings, barely less
# accurate on the whole test set (0.852 against 0.869 yesterday; here, 0.844 against
# 0.883: on 300 requests, the gap varies from one draw to another). The LLM (Qwen3-4B on
# a GPU: 0.754 zero-shot, 0.763 few-shot) stays well behind them, among the TF-IDF
# methods (0.691 to 0.821), which 300 requests don't tell apart, for 302 and 331 ms per
# text: hundreds of times slower than the encoder. For a prototype without labelled
# data: the LLM, right away.

## 4. Read the Errors

The method of the "Reading the Errors" slide, on the fine-tuned encoder's errors among the 2,974 test requests. Its logits are published (`massive-intents/lab-results/encoder-logits-en.npy`, already downloaded: one row per request of `test`, one column per intent, in alphabetical order). They are our encoder's, trained on a GPU, not yours, of which yesterday's lab only kept the predictions: the exercise is the same with or without a GPU.

Turn the logits into probabilities (`softmax`), keep the errors and sort them by decreasing confidence. Read the first 10, with the intents' descriptions (`intents`): model error, or wrong label? Then group the errors by pair (expected intent, predicted intent): which pairs come up most often, and what would you do about them?

**Useful documentation**

- [`numpy.load`](https://numpy.org/doc/stable/reference/generated/numpy.load.html)
- [`scipy.special.softmax`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html)
- [`DataFrame.assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
- [`DataFrame.sort_values`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)
- [`DataFrame.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.value_counts.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `np.load(path)`: the logits, 2,974 rows (in `test`'s order) and 60 columns; `np.array(sorted(train.label.unique()))`: the intents, in the columns' order
- `softmax(logits, axis=1)`: one probability distribution per request
- `proba.argmax(axis=1)`: the number of the predicted intent, which `classes[...]` turns into an intent; `proba.max(axis=1)`: its probability, the model's confidence
- `test.assign(predicted=..., confidence=...)`: `test`, with two more columns
- `encoder[encoder.predicted != encoder.label]`: the errors (you should find 340), then `.sort_values("confidence", ascending=False)`: the surest first
- `errors.value_counts(["label", "predicted"])`: the number of errors of each pair, from the most frequent to the rarest
- `intents.set_index("label").description_en`: each intent's description, to decide between the model and the label
- `pd.set_option("display.max_colwidth", None)`: pandas shows the texts in full; `display(table)` shows a table in the middle of a cell

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import numpy as np
from IPython.display import display
from scipy.special import softmax

pd.set_option("display.max_colwidth", None)  # the texts in full
logits = np.load("massive-intents/lab-results/encoder-logits-en.npy")
classes = np.array(sorted(train.label.unique()))  # the order of the logits' columns
proba = softmax(..., axis=1)
encoder = test.assign(predicted=classes[proba.argmax(axis=1)], confidence=...)
errors = encoder[encoder.predicted != ...].sort_values("confidence", ascending=...)
print(len(errors), "errors,", (errors.confidence > 0.99).sum(), "above 0.99")
# the columns: the text, the label, the prediction and the confidence
display(errors[[..., ..., ..., ...]].head(10))
errors.value_counts([..., ...]).head(10)
```

### Solution

In [ ]:
import numpy as np
from IPython.display import display
from scipy.special import softmax

pd.set_option("display.max_colwidth", None)  # the texts in full
logits = np.load("massive-intents/lab-results/encoder-logits-en.npy")
classes = np.array(sorted(train.label.unique()))  # the order of the logits' columns
proba = softmax(logits, axis=1)
encoder = test.assign(
  predicted=classes[proba.argmax(axis=1)], confidence=proba.max(axis=1)
)
errors = encoder[encoder.predicted != encoder.label].sort_values(
  "confidence", ascending=False
)
print(len(errors), "errors,", (errors.confidence > 0.99).sum(), "above 0.99 confidence")
display(errors[["text", "label", "predicted", "confidence"]].head(10))
errors.value_counts(["label", "predicted"]).head(10)

# 340 errors, 81 of them above 0.99 confidence. Among the 10 surest, most labels are
# wrong, as the slide says: a stock price ("get me the latest stock price for costco")
# or an exchange rate is labelled news_query instead of qa_stock or qa_currency, "what
# date is it" calendar_query instead of datetime_query, the traffic on a commute
# transport_query instead of transport_traffic. A few errors are the model's: "be more
# chill by ned vizzini" is a novel, so play_audiobook was right. The most frequent
# pairs, qa_factoid and general_quirky (11 and 10 errors), and general_quirky taken for
# news_query (11): intents with blurred boundaries, general_quirky being "outside the
# other categories". Defining them better, or merging them, pays more than a bigger
# model.